# 01 · Collecting and cleaning Tiki data

**Goal:** collect the products Tiki lists for a keyword, understand the raw data, and turn it into a clean table that is ready for analysis.

1. Is it OK to collect this data?
2. First try: the search page with `requests` + BeautifulSoup
3. Second way: the JSON API the page calls
4. Collect 5 pages and look at the raw data
5. Clean it, step by step
6. Build the sample dataset used in notebook 02

> Sections 1-5 download live data, so the numbers change a little every time the notebook runs.

In [1]:
import json
import sys
import time
import urllib.robotparser
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

# make the project files (scraper.py, cleaning.py) importable from the notebooks/ folder
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import scraper
from cleaning import clean_products, main_category, matches_keyword

KEYWORD = "chuột gaming"
pd.set_option("display.max_colwidth", 60)

## 1. Is it OK to collect this data?

Before writing a scraper I check:

- **robots.txt**: does Tiki ask bots not to visit these pages?
- **Access**: no login, no cookie, only public product information (name, price, number sold, rating).
- **Load**: a few pages only, with a pause of 1 second between two requests.

In [2]:
robots = urllib.robotparser.RobotFileParser("https://tiki.vn/robots.txt")
robots.read()
for url in ["https://tiki.vn/search?q=chuot+gaming", "https://tiki.vn/api/v2/products?q=chuot+gaming"]:
    print("allowed" if robots.can_fetch("*", url) else "NOT allowed", "->", url)

allowed -> https://tiki.vn/search?q=chuot+gaming
allowed -> https://tiki.vn/api/v2/products?q=chuot+gaming


Both are allowed. The `Disallow` lines of Tiki's robots.txt only block spam searches (casino and betting words, `.com` domains...), not normal product searches.

## 2. First try: the search page with requests + BeautifulSoup

In Chrome (DevTools > Elements) every product on `tiki.vn/search?q=...` is a card `<a data-view-id="product_list_item">`. Let's download the page with `requests` and look for those cards with BeautifulSoup.

In [3]:
response = requests.get(scraper.SEARCH_PAGE_URL, params={"q": KEYWORD}, headers=scraper.HEADERS, timeout=15)
soup = BeautifulSoup(response.text, "html.parser")

print("Status:", response.status_code, "| page size:", round(len(response.content) / 1024), "KB")
print("Title:", soup.title.get_text())
print("Product cards found:", len(soup.select('a[data-view-id="product_list_item"]')))

Status: 200 | page size: 659 KB
Title: Chuột gaming hàng chính hãng, giao nhanh - Tháng 10, 2026 | Mua Ngay | tiki.vn
Product cards found: 0


**0 cards.** The cards are drawn by JavaScript *in the browser*, after the page loads; `requests` only gets the HTML before that happens.

But *View page source* (Ctrl+U) shows a `<script id="__NEXT_DATA__">` tag: Tiki's website is built with Next.js, which writes all the data of the page into that tag as JSON. BeautifulSoup can find the tag, and `json.loads` can read it.

In [4]:
script = soup.find("script", id="__NEXT_DATA__")
page_data = json.loads(script.string)
items = page_data["props"]["initialState"]["catalog"]["data"]
ads = [item for item in items if item.get("advertisement")]

print("Products inside __NEXT_DATA__:", len(items))
print("of which sponsored (ads):", len(ads))

# the same steps, as written in scraper.py
html_items = scraper.parse_search_page(response.text)
print("Kept by scraper.parse_search_page():", len(html_items))

Products inside __NEXT_DATA__: 58
of which sponsored (ads): 18
Kept by scraper.parse_search_page(): 40


## 3. Second way: the JSON API

DevTools > Network > Fetch/XHR shows that the search page also calls `tiki.vn/api/v2/products?q=...&page=...`: an API that answers with JSON directly. `scraper.get_api_page()` calls it (and waits then retries if Tiki answers *429 Too Many Requests*).

In [5]:
data = scraper.get_api_page(KEYWORD, page=1)
api_items = data["data"]
paging = data["paging"]

print("Results:", paging["total"], "| pages:", paging["last_page"], "| per page:", paging["per_page"])
print("Products on page 1:", len(api_items))
print("Same products as the HTML page:", {item["id"] for item in api_items} == {item["id"] for item in html_items})

Results: 1892 | pages: 48 | per page: 40
Products on page 1: 40
Same products as the HTML page: True


In [6]:
# Download page 1 both ways and compare
rows = []
for method, url, params in [
    ("HTML page + BeautifulSoup", scraper.SEARCH_PAGE_URL, {"q": KEYWORD}),
    ("JSON API", scraper.API_URL, {"q": KEYWORD, "limit": 40}),
]:
    start = time.time()
    r = requests.get(url, params=params, headers=scraper.HEADERS, timeout=15)
    rows.append({"method": method, "KB downloaded": round(len(r.content) / 1024),
                 "seconds": round(time.time() - start, 2)})
    time.sleep(scraper.DELAY)
pd.DataFrame(rows).set_index("method")

,KB downloaded,seconds
method,,
HTML page + BeautifulSoup,659,0.73
JSON API,160,0.41


Both ways give **the same products**. The pipeline (and the web app) uses the API because:

- it downloads much less (see the table: only the products, not the whole page),
- the number of pages (`paging`) comes with the answer,
- there are no ads to remove, and no HTML structure that can change when Tiki redesigns its website.

BeautifulSoup stays the right tool for websites that have no such API: then the data must be read from the HTML itself.

## 4. Collect 5 pages and look at the raw data

One product from the API has about 45 fields. `scraper.parse_item()` keeps the 11 that answer our questions:

In [7]:
item = api_items[0]
fields = ["id", "name", "brand_name", "primary_category_name", "price", "original_price",
          "quantity_sold", "rating_average", "review_count", "url_path"]
print(len(item), "fields in the raw item. A few of them:")
{field: item.get(field) for field in fields}

46 fields in the raw item. A few of them:


{'id': 56196502,
 'name': 'Chuột Gaming Logitech G102 Gen 2 Lightsync - Hàng Chính Hãng',
 'brand_name': 'Logitech',
 'primary_category_name': 'Chuột Chơi Game',
 'price': 471900,
 'original_price': 471900,
 'quantity_sold': {'text': 'Đã bán 4757', 'value': 4757},
 'rating_average': 4.8,
 'review_count': 1392,
 'url_path': 'chuot-gaming-logitech-g102-gen-2-lightsync-hang-chinh-hang-p56196502.html?spid=73161623'}

| API field | Column | Note |
|---|---|---|
| `id` | `product_id` | unique id on Tiki |
| `name`, `brand_name` | `name`, `brand` | |
| `primary_category_name` | `category` | e.g. *Chuột Chơi Game* |
| `price`, `original_price` | `price`, `original_price` | VND |
| `quantity_sold.value` | `sold` | missing when nothing was sold yet → 0 |
| `rating_average`, `review_count` | `rating`, `review_count` | rating 0 = not rated yet |
| `thumbnail_url` | `image_url` | |
| `url_path` | `product_url` | `https://tiki.vn/` + `url_path` |

`scraper.crawl()` repeats this for up to 5 pages (200 products), with a pause between two pages.

In [8]:
raw = scraper.crawl(KEYWORD, pages=5)
print(raw.shape)
raw.head(3)

(200, 11)


,product_id,name,brand,category,price,original_price,sold,rating,review_count,image_url,product_url
0,56196502,Chuột Gaming Logitech G102 Gen 2 Lightsync - Hàng Chính ...,Logitech,Chuột Chơi Game,471900,471900,4757,4.8,1392,https://salt.tikicdn.com/cache/280x280/ts/product/f8/39/...,https://tiki.vn/chuot-gaming-logitech-g102-gen-2-lightsy...
1,276611072,Chuột Gaming Có Dây Newmen Game G10+ Hàng Chính Hãng,Newmen,Chuột Chơi Game,199000,199000,18,5.0,1,https://salt.tikicdn.com/cache/280x280/ts/product/87/a6/...,https://tiki.vn/chuot-gaming-co-day-newmen-game-g10-hang...
2,195854829,Chuột Gaming E-DRA EM6102w - Hàng Chính Hãng,E-DRA,Chuột Chơi Game,243000,243000,6,0.0,0,https://salt.tikicdn.com/cache/280x280/ts/product/0e/ff/...,https://tiki.vn/chuot-gaming-e-dra-em6102w-p195854829.ht...


In [9]:
# Data quality: share of empty values and of zeros, per column
quality = pd.DataFrame({
    "empty %": (raw.isna().mean() * 100).round(1),
    "zero %": ((raw == 0).mean() * 100).round(1),
})
quality[(quality["empty %"] > 0) | (quality["zero %"] > 0)]

,empty %,zero %
sold,0.0,40.0
rating,0.0,63.5
review_count,0.0,63.5


What this table shows:

- **`rating` = 0** for many products. That means *not rated yet*, not *zero stars*: averaging those zeros would pull the average rating down (a bug of version 1).
- **`sold` = 0** for many products: they are listed but have never sold. That is real information, so it stays 0.
- **`brand`** is sometimes empty or "OEM": no brand.

## 5. Cleaning, step by step

The rules live in `cleaning.clean_products()`. Here the steps that remove rows are applied one by one, to see how many rows each one removes.

In [10]:
steps = {"raw rows": len(raw)}

df = raw.dropna(subset=["name", "product_url", "price"])
df = df[df["price"] > 0]
steps["with a name, link and price"] = len(df)

df = df.drop_duplicates(subset="product_id")
steps["without duplicates"] = len(df)

category = main_category(df, KEYWORD)
steps[f"in the main category ({category})"] = int((df["category"] == category).sum())

pd.Series(steps).to_frame("rows")

,rows
raw rows,200
"with a name, link and price",200
without duplicates,200
in the main category (Chuột Chơi Game),152


### Why keep only the main category?

A search for *chuột gaming* also returns mouse pads, keyboards and office mice. Comparing their prices with gaming mice would give a meaningless "fair price". So I keep one category: the one most products *whose name contains every word of the keyword* belong to (accent-insensitive, whole words).

In [11]:
exact = df["name"].apply(matches_keyword, keyword=KEYWORD)
in_category = df["category"] == category

print("Name contains every word of the keyword:", int(exact.sum()))
print("In the main category:                   ", int(in_category.sum()))
print("  of which kept although the name does not match (e.g. 'chuột chơi game'):", int((in_category & ~exact).sum()))
print("Removed (other categories):             ", int((~in_category).sum()))
df.loc[~in_category, ["name", "category"]].head(6)

Name contains every word of the keyword: 113
In the main category:                    152
  of which kept although the name does not match (e.g. 'chuột chơi game'): 60
Removed (other categories):              48


,name,category
37,Chuột led gaming macro X11 7200dpi chống mỏi cổ tay cho ...,Chuột Văn Phòng Không Dây
70,Lót Chuột Gaming HT SYS - Hàng Chính Hãng,Bàn Di Chuột - Miếng Lót Chuột
72,Chuột Led RGB gaming T91 7200dpi công thái học chống mỏi...,Chuột Văn Phòng Có Dây
76,Lót Chuột Gaming RGB 80x30 cm,Bàn Di Chuột - Miếng Lót Chuột
78,Chuột Không Dây T26 Gaming led RGB Bluetooth + Usb2.4G P...,Chuột Văn Phòng Không Dây
79,Chuột Led Không Dây Pin sạc T25 Gaming Cổng sạc TypeC ch...,Chuột Văn Phòng Không Dây


The rule is simple, not perfect: a gaming mouse that a seller listed under *office mice* is removed too. Category labels come from the sellers, so they are not always right.

### The clean table

In [12]:
clean = clean_products(raw, KEYWORD)
print(clean.shape)
clean.dtypes

(152, 12)


product_id         object
name               object
brand              object
category           object
price               int64
original_price      int64
discount_pct        int64
sold                int64
rating            float64
review_count        int64
image_url          object
product_url        object
dtype: object

In [13]:
clean[["price", "original_price", "discount_pct", "sold", "rating", "review_count"]].describe().round(1)

,price,original_price,discount_pct,sold,rating,review_count
count,152.0,152.0,152.0,152.0,59.0,152.0
mean,767219.3,809197.3,2.2,84.0,4.6,21.8
std,931688.8,1060982.0,7.4,496.5,0.8,129.6
min,22093.0,22093.0,0.0,0.0,1.0,0.0
25%,241000.0,244500.0,0.0,0.0,4.5,0.0
50%,398000.0,398000.0,0.0,1.0,5.0,0.0
75%,795250.0,799000.0,0.0,9.2,5.0,1.2
max,5359000.0,7149000.0,50.0,4757.0,5.0,1392.0


## 6. The sample dataset

Three keywords from different kinds of products, collected with the same code, 30 seconds apart. Notebook 02 analyses this file.

Set `REBUILD_SAMPLE = True` to collect it again; by default the saved file is used, so that the numbers in notebook 02 and in the README do not change every time this notebook runs.

In [14]:
REBUILD_SAMPLE = False   # True: collect again and overwrite data/tiki_sample.csv
KEYWORDS = ["chuột gaming", "tai nghe bluetooth", "nồi chiên không dầu"]
sample_path = ROOT / "data" / "tiki_sample.csv"

if REBUILD_SAMPLE:
    tables = []
    for keyword in KEYWORDS:
        products = clean_products(scraper.crawl(keyword, pages=5), keyword)
        products.insert(0, "keyword", keyword)
        tables.append(products)
        time.sleep(30)  # be polite between two keywords
    sample = pd.concat(tables, ignore_index=True)
    sample["crawled_at"] = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")
    sample.to_csv(sample_path, index=False, encoding="utf-8-sig")

sample = pd.read_csv(sample_path)
print(len(sample), "products, collected on", sample["crawled_at"].iloc[0])
sample.groupby("keyword", sort=False)["category"].agg(category="first", products="size")

410 products, collected on 2026-10-25 18:07


,category,products
keyword,,
chuột gaming,Chuột Chơi Game,152
tai nghe bluetooth,Tai Nghe True Wireless,70
nồi chiên không dầu,Nồi chiên,188


**Next:** `02_price_analysis.ipynb` analyses this sample.